# Pipecat Smart Turn — Training

Fine-tune `openai/whisper-tiny` encoder with an attention-pooling classification
head for binary turn detection (complete vs incomplete).

Based on [pipecat-ai/smart-turn](https://github.com/pipecat-ai/smart-turn) `train.py`.

**Run inside the `smart-turn` Docker container on the GPU VM.**

In [ ]:
import os
import numpy as np
import torch
import torch.nn as nn
from datasets import load_dataset
from transformers import (
    WhisperConfig,
    WhisperFeatureExtractor,
    WhisperPreTrainedModel,
    Trainer,
    TrainingArguments,
)
from transformers.models.whisper.modeling_whisper import WhisperEncoder
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from torch.utils.data import Dataset

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
if device.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

## 1. Config

In [ ]:
BASE_MODEL = "openai/whisper-tiny"
CHUNK_LENGTH = 8          # seconds — model input cap
SAMPLE_RATE = 16_000

# Training — tuned for T4 16 GB VRAM
# effective batch = BATCH_SIZE * GRAD_ACCUM = 32 * 12 = 384 (matches upstream)
BATCH_SIZE = 32
GRAD_ACCUM = 12
EVAL_BATCH_SIZE = 64
EPOCHS = 4
LR = 5e-5
WARMUP_RATIO = 0.2
WEIGHT_DECAY = 0.01
EVAL_STEPS = 500
SAVE_STEPS = 500
DATALOADER_WORKERS = 6

# ONNX
ONNX_OPSET = 18
CALIBRATION_SAMPLES = 1024

# Paths
RUN_NAME = "wavekat-v1"
OUTPUT_DIR = f"/checkpoints/{RUN_NAME}"
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"Output: {OUTPUT_DIR}")

## 2. Model

Whisper Tiny encoder → attention pooling → classifier head (single logit).

In [ ]:
class SmartTurnModel(WhisperPreTrainedModel):
    """Whisper encoder + attention pooling + binary classifier."""

    def __init__(self, config: WhisperConfig):
        super().__init__(config)
        # Override max positions for 8s input (800 frames / 2 = 400 encoder steps)
        config.max_source_positions = 400
        self.encoder = WhisperEncoder(config)

        hidden = config.d_model  # 384 for whisper-tiny

        # Attention pooling over time
        self.pool_attention = nn.Sequential(
            nn.Linear(hidden, 256),
            nn.Tanh(),
            nn.Linear(256, 1),
        )

        # Classification head
        self.classifier = nn.Sequential(
            nn.Linear(hidden, 256),
            nn.LayerNorm(256),
            nn.GELU(),
            nn.Dropout(0.1),
            nn.Linear(256, 64),
            nn.GELU(),
            nn.Linear(64, 1),
        )

        self.post_init()

    def forward(self, input_features, labels=None):
        enc = self.encoder(input_features).last_hidden_state  # (B, T, H)

        # Attention pooling
        attn_weights = self.pool_attention(enc).squeeze(-1)        # (B, T)
        attn_weights = torch.softmax(attn_weights, dim=-1)         # (B, T)
        pooled = torch.bmm(attn_weights.unsqueeze(1), enc).squeeze(1)  # (B, H)

        logits = self.classifier(pooled).squeeze(-1)  # (B,)
        probs = torch.sigmoid(logits)

        loss = None
        if labels is not None:
            pos_weight = ((labels == 0).sum() / (labels == 1).sum().clamp(min=1)).clamp(0.1, 10.0)
            loss_fn = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
            loss = loss_fn(logits, labels.float())

        return {"loss": loss, "logits": probs}


print("Model class defined.")

In [ ]:
config = WhisperConfig.from_pretrained(BASE_MODEL)
model = SmartTurnModel(config)

# Load pretrained encoder weights (ignore missing classifier/pool keys)
pretrained = SmartTurnModel.from_pretrained(BASE_MODEL, config=config, ignore_mismatched_sizes=True)
model.encoder.load_state_dict(pretrained.encoder.state_dict())
del pretrained

total = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Parameters: {total:,} total, {trainable:,} trainable")

## 3. Dataset

On-demand mel spectrogram extraction — avoids materialising all features in RAM.

In [ ]:
def truncate_to_last_n_seconds(audio_array: np.ndarray, sr: int, n: int = 8) -> np.ndarray:
    """Keep the last n seconds; zero-pad at the start if shorter."""
    max_samples = sr * n
    if len(audio_array) > max_samples:
        return audio_array[-max_samples:]
    elif len(audio_array) < max_samples:
        pad = np.zeros(max_samples - len(audio_array), dtype=audio_array.dtype)
        return np.concatenate([pad, audio_array])
    return audio_array


class SmartTurnDataset(Dataset):
    """Wraps a HuggingFace dataset, extracting mel features on the fly."""

    def __init__(self, hf_dataset, feature_extractor):
        self.ds = hf_dataset
        self.fe = feature_extractor

    def __len__(self):
        return len(self.ds)

    def __getitem__(self, idx):
        sample = self.ds[idx]
        audio = sample["audio"]
        arr = np.array(audio["array"], dtype=np.float32)
        arr = truncate_to_last_n_seconds(arr, audio["sampling_rate"], CHUNK_LENGTH)

        features = self.fe(
            arr,
            sampling_rate=SAMPLE_RATE,
            return_tensors="pt",
            padding="max_length",
            max_length=CHUNK_LENGTH * SAMPLE_RATE,
            truncation=True,
            do_normalize=True,
        )
        return {
            "input_features": features["input_features"].squeeze(0),  # (80, 800)
            "labels": torch.tensor(float(sample["endpoint_bool"])),
        }


feature_extractor = WhisperFeatureExtractor(chunk_length=CHUNK_LENGTH)
print("Dataset class + feature extractor ready.")

In [ ]:
ds_train_raw = load_dataset("pipecat-ai/smart-turn-data-v3.2-train", split="train")
ds_test_raw = load_dataset("pipecat-ai/smart-turn-data-v3.2-test", split="train")

# Split training data 90/10 for train/eval (matches upstream)
split = ds_train_raw.train_test_split(test_size=0.1, seed=42)
ds_train_split = split["train"]
ds_eval_split = split["test"]

train_dataset = SmartTurnDataset(ds_train_split, feature_extractor)
eval_dataset = SmartTurnDataset(ds_eval_split, feature_extractor)
test_dataset = SmartTurnDataset(ds_test_raw, feature_extractor)

print(f"Train: {len(train_dataset):,}")
print(f"Eval:  {len(eval_dataset):,}")
print(f"Test:  {len(test_dataset):,}")

In [ ]:
# Sanity check — verify a single sample shape
sample = train_dataset[0]
print(f"input_features: {sample['input_features'].shape}")  # expect (80, 800)
print(f"label: {sample['labels']}")

## 4. Train

In [ ]:
def compute_metrics(eval_pred):
    probs, labels = eval_pred
    preds = (probs > 0.5).astype(int).flatten()
    labels = labels.astype(int).flatten()
    return {
        "accuracy": accuracy_score(labels, preds),
        "precision": precision_score(labels, preds, zero_division=0),
        "recall": recall_score(labels, preds, zero_division=0),
        "f1": f1_score(labels, preds, zero_division=0),
    }


training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    run_name=RUN_NAME,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=EVAL_BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LR,
    warmup_ratio=WARMUP_RATIO,
    weight_decay=WEIGHT_DECAY,
    lr_scheduler_type="cosine",
    eval_strategy="steps",
    eval_steps=EVAL_STEPS,
    save_steps=SAVE_STEPS,
    logging_steps=100,
    dataloader_num_workers=DATALOADER_WORKERS,
    dataloader_prefetch_factor=4,
    bf16=torch.cuda.is_bf16_supported() if torch.cuda.is_available() else False,
    fp16=not torch.cuda.is_bf16_supported() if torch.cuda.is_available() else False,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    save_total_limit=3,
    report_to="wandb" if os.environ.get("WANDB_API_KEY") else "none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    compute_metrics=compute_metrics,
)

print(f"Effective batch size: {BATCH_SIZE * GRAD_ACCUM}")
print(f"Mixed precision: {'bf16' if training_args.bf16 else 'fp16' if training_args.fp16 else 'none'}")
print(f"W&B: {'enabled' if training_args.report_to == ['wandb'] else 'disabled'}")

In [ ]:
trainer.train()

## 5. Evaluate on Test Set

In [ ]:
test_results = trainer.evaluate(test_dataset, metric_key_prefix="test")
for k, v in sorted(test_results.items()):
    if isinstance(v, float):
        print(f"  {k}: {v:.4f}")

In [ ]:
# Save model + feature extractor
model.save_pretrained(OUTPUT_DIR)
feature_extractor.save_pretrained(OUTPUT_DIR)
print(f"Saved to {OUTPUT_DIR}")

## 6. ONNX Export (FP32)

In [ ]:
import onnx
import onnxruntime as ort


class ONNXWrapper(nn.Module):
    """Thin wrapper that reshapes output to (batch, 1) for ONNX consumers."""

    def __init__(self, model):
        super().__init__()
        self.model = model

    def forward(self, input_features):
        out = self.model(input_features)
        return out["logits"].unsqueeze(-1)


onnx_fp32_path = os.path.join(OUTPUT_DIR, "smart-turn-v3.onnx")

wrapper = ONNXWrapper(model).cpu().eval()
dummy = torch.randn(1, 80, 800)

torch.onnx.export(
    wrapper,
    (dummy,),
    onnx_fp32_path,
    opset_version=ONNX_OPSET,
    input_names=["input_features"],
    output_names=["logits"],
    dynamic_axes={"input_features": {0: "batch"}, "logits": {0: "batch"}},
    do_constant_folding=False,
)

onnx.checker.check_model(onnx.load(onnx_fp32_path))
print(f"FP32 ONNX: {onnx_fp32_path}")
print(f"Size: {os.path.getsize(onnx_fp32_path) / 1e6:.1f} MB")

# Quick verify
sess = ort.InferenceSession(onnx_fp32_path, providers=["CPUExecutionProvider"])
out = sess.run(None, {"input_features": dummy.numpy()})
print(f"Test output shape: {out[0].shape}, value: {out[0][0, 0]:.4f}")

## 7. INT8 Quantization

Static quantization with entropy calibration on 1024 training samples.

In [ ]:
from onnxruntime.quantization import (
    CalibrationMethod,
    QuantFormat,
    QuantType,
    quantize_static,
    quant_pre_process,
    CalibrationDataReader,
)


class SmartTurnCalibrationReader(CalibrationDataReader):
    """Feeds calibration samples to the ONNX quantizer."""

    def __init__(self, dataset, n_samples=1024):
        self.samples = []
        rng = np.random.default_rng(42)
        indices = rng.choice(len(dataset), size=min(n_samples, len(dataset)), replace=False)
        for i in indices:
            item = dataset[int(i)]
            self.samples.append({"input_features": item["input_features"].unsqueeze(0).numpy()})
        self.idx = 0

    def get_next(self):
        if self.idx >= len(self.samples):
            return None
        sample = self.samples[self.idx]
        self.idx += 1
        return sample

    def rewind(self):
        self.idx = 0


# Pre-process graph
pre_path = onnx_fp32_path.replace(".onnx", "-pre.onnx")
quant_pre_process(onnx_fp32_path, pre_path, skip_symbolic_shape=True)

# Quantize
onnx_int8_path = os.path.join(OUTPUT_DIR, "smart-turn-v3-int8.onnx")
reader = SmartTurnCalibrationReader(train_dataset, CALIBRATION_SAMPLES)

quantize_static(
    model_input=pre_path,
    model_output=onnx_int8_path,
    calibration_data_reader=reader,
    quant_format=QuantFormat.QDQ,
    activation_type=QuantType.QUInt8,
    weight_type=QuantType.QInt8,
    per_channel=True,
    calibrate_method=CalibrationMethod.Entropy,
    op_types_to_quantize=["Conv", "MatMul", "Gemm"],
)

# Clean up temp file
os.remove(pre_path)

print(f"INT8 ONNX: {onnx_int8_path}")
print(f"Size: {os.path.getsize(onnx_int8_path) / 1e6:.1f} MB")

# Verify
sess_int8 = ort.InferenceSession(onnx_int8_path, providers=["CPUExecutionProvider"])
out_int8 = sess_int8.run(None, {"input_features": dummy.numpy()})
print(f"INT8 test output: {out_int8[0][0, 0]:.4f} (FP32 was {out[0][0, 0]:.4f})")

## 8. Benchmark

Latency comparison: FP32 vs INT8 on CPU.

In [ ]:
import time

def benchmark_onnx(path, label, n_runs=100, warmup=10):
    sess = ort.InferenceSession(path, providers=["CPUExecutionProvider"])
    x = np.random.randn(1, 80, 800).astype(np.float32)
    for _ in range(warmup):
        sess.run(None, {"input_features": x})
    times = []
    for _ in range(n_runs):
        t0 = time.perf_counter()
        sess.run(None, {"input_features": x})
        times.append((time.perf_counter() - t0) * 1000)
    times = np.array(times)
    print(f"{label}: mean={times.mean():.2f}ms, p50={np.median(times):.2f}ms, p99={np.percentile(times, 99):.2f}ms")

benchmark_onnx(onnx_fp32_path, "FP32")
benchmark_onnx(onnx_int8_path, "INT8")

## 9. Sanity Check — Inference on Test Samples

In [ ]:
sess_int8 = ort.InferenceSession(onnx_int8_path, providers=["CPUExecutionProvider"])

print("Sample predictions (INT8 ONNX):\n")
for i in range(10):
    sample = test_dataset[i]
    inp = sample["input_features"].unsqueeze(0).numpy()
    prob = sess_int8.run(None, {"input_features": inp})[0][0, 0]
    label = "Complete" if sample["labels"] > 0.5 else "Incomplete"
    pred = "Complete" if prob > 0.5 else "Incomplete"
    match = "OK" if label == pred else "MISS"
    print(f"  [{i}] truth={label:<12} pred={pred:<12} prob={prob:.4f}  {match}")

---

**Artifacts in `OUTPUT_DIR`:**
- `smart-turn-v3.onnx` — FP32 (~32 MB)
- `smart-turn-v3-int8.onnx` — INT8 (~8 MB)
- `model.safetensors` + `config.json` — PyTorch checkpoint

Copy the INT8 model to integrate into wavekat-turn.